# 🔍 Root Finding Methods — Numerical Analysis I
**Reference:** Burden & Faires, *Numerical Analysis*, 9th ed. — Chapter 2

---
## 📚 Topics Covered
| # | Method | Convergence Order |
|---|--------|------------------|
| 1 | Bisection Method | Linear — O(1/2ⁿ) |
| 2 | Fixed-Point Iteration | Linear — depends on g'(p) |
| 3 | Newton-Raphson Method | Quadratic |
| 4 | Secant Method | Superlinear (~1.618) |
| 5 | False Position (Regula Falsi) | Linear |
| 6 | Convergence Comparison | All methods |

> **Goal:** Find a root $p$ such that $f(p) = 0$, given a continuous function $f$ on $[a, b]$.

In [ ]:
# ─────────────────────────────────────────────
# 📦 Imports & Setup
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tabulate import tabulate  # pip install tabulate

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 12
})
print('✅ Setup complete.')

---
## 1️⃣ Bisection Method

**Idea:** If $f(a) \cdot f(b) < 0$, the Intermediate Value Theorem guarantees a root in $[a, b]$.
Repeatedly halve the interval, keeping the sub-interval that contains the sign change.

$$p_n = \frac{a_n + b_n}{2}$$

**Error bound:** $|p_n - p| \leq \dfrac{b - a}{2^n}$

**Convergence:** Linear — guaranteed but slow.

In [ ]:
def bisection(f, a, b, tol=1e-7, max_iter=100):
    """
    Bisection Method for root finding.

    Parameters:
    -----------
    f        : callable — the function f(x)
    a, b     : float    — initial bracket [a, b] where f(a)*f(b) < 0
    tol      : float    — tolerance for stopping criterion
    max_iter : int      — maximum number of iterations

    Returns:
    --------
    root     : float    — approximate root
    history  : list     — list of (iteration, midpoint, f(midpoint), error)
    """
    # Validate bracket
    if f(a) * f(b) > 0:
        raise ValueError("f(a) and f(b) must have opposite signs.")

    history = []

    for n in range(1, max_iter + 1):
        p = (a + b) / 2          # midpoint
        fp = f(p)
        error = (b - a) / 2      # guaranteed error bound

        history.append((n, p, fp, error))

        # Stopping criterion
        if abs(fp) < tol or error < tol:
            return p, history

        # Narrow the bracket
        if f(a) * fp < 0:
            b = p   # root is in [a, p]
        else:
            a = p   # root is in [p, b]

    print(f"⚠️  Maximum iterations ({max_iter}) reached.")
    return p, history

In [ ]:
# ── Example 1: f(x) = x³ - x - 2  →  root near x = 1.5214 ──
f1 = lambda x: x**3 - x - 2

root_b, hist_b = bisection(f1, a=1, b=2, tol=1e-6)

# Display iteration table
df_b = pd.DataFrame(hist_b, columns=['Iteration', 'Midpoint pₙ', 'f(pₙ)', 'Error Bound'])
print(f"\n📌 Bisection: f(x) = x³ - x - 2  on [1, 2]")
print(f"   Root ≈ {root_b:.10f}")
print(f"   Converged in {len(hist_b)} iterations\n")
print(df_b.to_string(index=False, float_format='{:.8f}'.format))

---
## 2️⃣ Fixed-Point Iteration

**Idea:** Rewrite $f(x) = 0$ as $x = g(x)$. Iterate:
$$p_{n+1} = g(p_n)$$

**Convergence condition (Fixed-Point Theorem):**
If $|g'(x)| \leq k < 1$ on $[a, b]$, then the iteration converges to a unique fixed point.

**Error bound:** $|p_n - p| \leq \dfrac{k^n}{1-k}|p_1 - p_0|$

In [ ]:
def fixed_point_iteration(g, p0, tol=1e-7, max_iter=100):
    """
    Fixed-Point Iteration.

    Parameters:
    -----------
    g        : callable — iteration function g(x), where x = g(x)
    p0       : float    — initial guess
    tol      : float    — convergence tolerance |p_{n+1} - p_n| < tol
    max_iter : int      — maximum iterations

    Returns:
    --------
    root     : float
    history  : list of (n, p_n, |p_n - p_{n-1}|)
    """
    history = [(0, p0, None)]

    for n in range(1, max_iter + 1):
        p1 = g(p0)
        err = abs(p1 - p0)
        history.append((n, p1, err))

        if err < tol:
            return p1, history

        p0 = p1

    print(f"⚠️  Max iterations reached.")
    return p0, history


# ── Example: f(x) = x³ - x - 2 → g(x) = (x + 2)^(1/3) ──
g1 = lambda x: (x + 2) ** (1/3)

root_fp, hist_fp = fixed_point_iteration(g1, p0=1.5, tol=1e-7)

df_fp = pd.DataFrame(hist_fp, columns=['n', 'pₙ', '|pₙ - pₙ₋₁|'])
print(f"\n📌 Fixed-Point Iteration: g(x) = (x+2)^(1/3)")
print(f"   Root ≈ {root_fp:.10f}")
print(f"   Converged in {len(hist_fp)-1} iterations\n")
print(df_fp.to_string(index=False, float_format='{:.10f}'.format))

---
## 3️⃣ Newton-Raphson Method

**Derivation:** Use first-order Taylor expansion around current estimate $p_n$:
$$0 \approx f(p_n) + f'(p_n)(p - p_n) \implies p_{n+1} = p_n - \frac{f(p_n)}{f'(p_n)}$$

**Convergence:** Quadratic — error roughly squares at each step.
**Weakness:** Requires $f'(p_n) \neq 0$ and a good initial guess.

In [ ]:
def newton_raphson(f, df, p0, tol=1e-7, max_iter=50):
    """
    Newton-Raphson Method.

    Parameters:
    -----------
    f        : callable — function f(x)
    df       : callable — derivative f'(x)
    p0       : float    — initial guess
    tol      : float    — tolerance on |p_{n+1} - p_n|
    max_iter : int      — maximum iterations

    Returns:
    --------
    root    : float
    history : list of (n, p_n, f(p_n), |update|)
    """
    history = [(0, p0, f(p0), None)]

    for n in range(1, max_iter + 1):
        fpn = f(p0)
        dfpn = df(p0)

        if abs(dfpn) < 1e-14:
            raise ZeroDivisionError(f"f'(p) ≈ 0 at p = {p0}. Choose a different initial guess.")

        p1 = p0 - fpn / dfpn    # Newton update
        err = abs(p1 - p0)
        history.append((n, p1, f(p1), err))

        if err < tol:
            return p1, history

        p0 = p1

    print("⚠️  Max iterations reached.")
    return p0, history


# ── Example: f(x) = x³ - x - 2,  f'(x) = 3x² - 1 ──
f1  = lambda x: x**3 - x - 2
df1 = lambda x: 3*x**2 - 1

root_nr, hist_nr = newton_raphson(f1, df1, p0=1.5, tol=1e-10)

df_nr = pd.DataFrame(hist_nr, columns=['n', 'pₙ', 'f(pₙ)', '|pₙ - pₙ₋₁|'])
print(f"\n📌 Newton-Raphson: f(x) = x³ - x - 2")
print(f"   Root ≈ {root_nr:.12f}")
print(f"   Converged in {len(hist_nr)-1} iterations\n")
print(df_nr.to_string(index=False, float_format='{:.12f}'.format))

---
## 4️⃣ Secant Method

**Idea:** Approximate $f'(p_n)$ by a finite difference — no derivative required!

$$p_{n+1} = p_n - f(p_n) \cdot \frac{p_n - p_{n-1}}{f(p_n) - f(p_{n-1})}$$

**Convergence order:** Superlinear — $\alpha \approx 1.618$ (the golden ratio!)

In [ ]:
def secant_method(f, p0, p1, tol=1e-7, max_iter=50):
    """
    Secant Method.

    Parameters:
    -----------
    f        : callable — function f(x)
    p0, p1   : float    — two initial approximations
    tol      : float    — convergence tolerance
    max_iter : int      — maximum iterations

    Returns:
    --------
    root     : float
    history  : list of (n, p_n, f(p_n), |update|)
    """
    history = [(0, p0, f(p0), None), (1, p1, f(p1), abs(p1 - p0))]

    for n in range(2, max_iter + 2):
        fp0, fp1 = f(p0), f(p1)

        if abs(fp1 - fp0) < 1e-14:
            raise ZeroDivisionError("f(p1) - f(p0) ≈ 0. Method fails.")

        p2 = p1 - fp1 * (p1 - p0) / (fp1 - fp0)   # secant update
        err = abs(p2 - p1)
        history.append((n, p2, f(p2), err))

        if err < tol:
            return p2, history

        p0, p1 = p1, p2

    print("⚠️  Max iterations reached.")
    return p1, history


# ── Example ──
root_sec, hist_sec = secant_method(f1, p0=1.0, p1=2.0, tol=1e-10)

df_sec = pd.DataFrame(hist_sec, columns=['n', 'pₙ', 'f(pₙ)', '|pₙ - pₙ₋₁|'])
print(f"\n📌 Secant Method: f(x) = x³ - x - 2")
print(f"   Root ≈ {root_sec:.12f}")
print(f"   Converged in {len(hist_sec)-2} iterations\n")
print(df_sec.to_string(index=False, float_format='{:.12f}'.format))

---
## 5️⃣ False Position Method (Regula Falsi)

Combines the **bracket guarantee** of Bisection with the **secant line** idea.
The next approximation is the x-intercept of the line through $(a, f(a))$ and $(b, f(b))$:

$$p = b - f(b)\frac{b-a}{f(b)-f(a)}$$

The bracket $[a, b]$ is updated to keep opposite signs — root always stays bracketed.

In [ ]:
def false_position(f, a, b, tol=1e-7, max_iter=100):
    """
    False Position (Regula Falsi) Method.

    Parameters:
    -----------
    f        : callable — function f(x)
    a, b     : float    — bracket [a, b] with f(a)*f(b) < 0
    tol      : float    — tolerance
    max_iter : int      — maximum iterations

    Returns:
    --------
    root     : float
    history  : list
    """
    if f(a) * f(b) > 0:
        raise ValueError("f(a) and f(b) must have opposite signs.")

    history = []
    for n in range(1, max_iter + 1):
        fa, fb = f(a), f(b)
        # False position formula: x-intercept of secant
        p = b - fb * (b - a) / (fb - fa)
        fp = f(p)
        err = abs(b - a)
        history.append((n, p, fp, err))

        if abs(fp) < tol:
            return p, history

        # Maintain bracket with opposite signs
        if fa * fp < 0:
            b = p
        else:
            a = p

    return p, history


root_fp2, hist_fp2 = false_position(f1, 1, 2, tol=1e-10)
print(f"\n📌 False Position: f(x) = x³ - x - 2")
print(f"   Root ≈ {root_fp2:.12f}")
print(f"   Converged in {len(hist_fp2)} iterations")

---
## 6️⃣ Convergence Comparison

Visualize how fast each method converges to the true root.

In [ ]:
# ── Collect errors from each method ──
TRUE_ROOT = 1.5213797068045675   # known root of x³ - x - 2

errors_bisection = [abs(p - TRUE_ROOT) for _, p, _, _ in hist_b]
errors_newton    = [abs(p - TRUE_ROOT) for _, p, _, _ in hist_nr]
errors_secant    = [abs(p - TRUE_ROOT) for _, p, _, _ in hist_sec]
errors_fpi       = [abs(p - TRUE_ROOT) for _, p, _ in hist_fp]

fig, ax = plt.subplots(figsize=(11, 6))

ax.semilogy(errors_bisection, 'o-', label='Bisection (Linear)', linewidth=2)
ax.semilogy(errors_fpi[1:], 's-', label='Fixed-Point (Linear)', linewidth=2)
ax.semilogy(errors_secant[1:], '^-', label='Secant (Superlinear ≈1.618)', linewidth=2)
ax.semilogy(errors_newton[1:], 'D-', label='Newton-Raphson (Quadratic)', linewidth=2)

ax.set_xlabel('Iteration Number')
ax.set_ylabel('Absolute Error  |pₙ - p*|  (log scale)')
ax.set_title('Convergence Comparison of Root-Finding Methods\nf(x) = x³ - x - 2', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# ── Summary Table ──
print("\n" + "="*55)
print(f"{'Method':<25} {'Iterations':>12} {'Final Error':>15}")
print("="*55)
print(f"{'Bisection':<25} {len(errors_bisection):>12} {errors_bisection[-1]:>15.2e}")
print(f"{'Fixed-Point':<25} {len(errors_fpi)-1:>12} {errors_fpi[-1]:>15.2e}")
print(f"{'Secant':<25} {len(errors_secant)-2:>12} {errors_secant[-1]:>15.2e}")
print(f"{'Newton-Raphson':<25} {len(errors_newton)-1:>12} {errors_newton[-1]:>15.2e}")
print("="*55)

---
## 🧪 Practice Problems

Try applying each method to the following:

1. $f(x) = \cos(x) - x$ on $[0, \pi/2]$ — (Cosine fixed point)
2. $f(x) = e^x - 3x$ on $[0, 2]$ — (two roots!)
3. $f(x) = x^4 - 3x^2 + 2$ — (multiple roots)
4. $f(x) = \sin(x) - x/2$ on $[\pi/2, \pi]$

**Challenge:** For Newton-Raphson on $f(x) = x^3 - 2x + 2$, start at $p_0 = 0$ and observe what happens.